# Script to quickly check model parameters

In [1]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts

In [10]:
path_to_model = "scripts/ModelShapValues/kfzteile24-aud-6835d77813526dc3d1229978/2026-09-09_new_models_current_image/2026-09-09_classifier_XGBRegressor_100_3_0.1_0_saved_model_cv_2_group_6835d77813526dc3d1229978_treatment.json"
#path_to_model = "scripts/ModelShapValues/kfzteile24-aud-6835d77813526dc3d1229978/2026-09-07_new_models_with_landingpage_lstm/2026-09-07_classifier_NNLSTM_50_nan_nan_nan_saved_model_cv_1_group_6835d77813526dc3d1229978_control.h5"

In [11]:
from pathlib import Path
import json
import pickle
import pandas as pd
from keras.models import load_model
from xgboost import XGBRegressor
model_path = Path(path_to_model)
inputs = (
    pd.read_parquet(model_path.parent / "inputs.parquet")["input"]
    .dropna()
    .tolist()
)
if model_path.suffix == ".h5" or "NNLSTM" in model_path.name:
    print("LSTM model")
    model = load_model(path_to_model, compile=False)
    
    params = {
        "path": path_to_model,
        "inputs": inputs,
        "input_shapes": [tuple(tensor.shape) for tensor in model.inputs],
        "layers": [
            {
                "name": layer.name,
                "type": layer.__class__.__name__,
                "params": layer.count_params(),
                "config": layer.get_config(),
            }
            for layer in model.layers
        ],
        "total_params": model.count_params(),
        "model_config": model.get_config(),
    }
elif "xgb" in str(path_to_model.lower()):
    print("XGBoost model")
    # XGBoost: .json via load_model, pickle (conversion/propensity) via pickle
    if model_path.suffix == ".json":
        model = XGBRegressor(enable_categorical=True)
        model.load_model(path_to_model)
    else:
        with open(path_to_model, "rb") as handle:
            model = pickle.load(handle)
    booster = model.get_booster()
    feature_names = booster.feature_names
    params = {
        "path": path_to_model,
        "inputs": feature_names or inputs,
        "n_features": booster.num_features(),
        "get_params": model.get_params(),
        "get_xgb_params": model.get_xgb_params(),
        "booster_config": json.loads(booster.save_config()),
    }

params